# 02 — Feature engineering

Same transformations as [`build_features`](../src/features/build_features.py): numeric **TotalCharges**, **num_active_services**, **tenure_group**. Below: distributions and **churn rate by bucket** to justify these features.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent))
from src.data.load_data import load_raw_data
from src.features.build_features import build_features
from src.data.preprocess import prepare_target
%matplotlib inline

In [ ]:
RAW = Path("../data/raw/telco_customer_churn.csv")
df_raw = load_raw_data(RAW)
df = build_features(df_raw.copy())
df["Churn"] = df_raw["Churn"].values
print("Rows:", len(df))
df[["tenure", "tenure_group", "num_active_services", "TotalCharges"]].head(12)

## `num_active_services` — count of active service flags

Higher counts can indicate engagement; relationship to churn is empirical.

In [ ]:
vc = df["num_active_services"].value_counts().sort_index()
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(vc.index.astype(str), vc.values, color="steelblue", edgecolor="black")
axes[0].set_xlabel("num_active_services")
axes[0].set_ylabel("Customers")
axes[0].set_title("Distribution")
rate = df.groupby("num_active_services")["Churn"].apply(lambda s: (s == "Yes").mean())
rate = rate.reindex(sorted(df["num_active_services"].unique()))
axes[1].bar(range(len(rate)), rate.values, color="coral", edgecolor="black")
axes[1].set_xticks(range(len(rate)))
axes[1].set_xticklabels(rate.index.astype(int))
axes[1].set_xlabel("num_active_services")
axes[1].set_ylabel("Churn rate")
axes[1].set_title("Churn rate by service count")
axes[1].axhline((df["Churn"] == "Yes").mean(), color="gray", ls="--", label="overall churn rate")
axes[1].legend()
plt.tight_layout()
plt.show()
print("Churn rate by num_active_services:\n", rate.round(3))

## `tenure_group` — tenure buckets

Aligns with business intuition: newer customers churn more often.

In [ ]:
order = ["0-12", "13-24", "25-48", "49-72"]
rate = df.groupby("tenure_group", observed=False)["Churn"].apply(lambda s: (s == "Yes").mean())
rate = rate.reindex([x for x in order if x in rate.index])
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(rate.index.astype(str), rate.values, color="#9b59b6", edgecolor="black")
ax.set_xlabel("tenure_group")
ax.set_ylabel("Churn rate")
ax.set_title("Churn rate by tenure bucket")
ax.axhline((df["Churn"] == "Yes").mean(), color="gray", ls="--", label="overall")
ax.legend()
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()
print(rate.round(3))

## `TotalCharges` after coercion

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
tc = df["TotalCharges"].dropna()
ax.hist(np.log1p(tc.clip(lower=0)), bins=40, color="teal", edgecolor="black", alpha=0.85)
ax.set_xlabel("log1p(TotalCharges)")
ax.set_title("TotalCharges (log) after numeric coercion")
plt.tight_layout()
plt.show()

## Feature matrix shape (matches training pipeline)

`customerID` dropped; target encoded 0/1 internally in `prepare_target`.

In [ ]:
df_model = build_features(df_raw)
X, y = prepare_target(df_model)
print("X shape:", X.shape, "| features:", X.shape[1])
print("Numeric:", list(X.select_dtypes(include=[np.number]).columns))
print("Categorical (sample):", [c for c in X.columns if c not in X.select_dtypes(include=[np.number]).columns][:5], "...")

## Leakage check

All features are known **at prediction time** (no future timestamps, no target-derived columns). `tenure_group` and `num_active_services` are deterministic functions of raw subscription fields.